# 单位规整化看板

选一个 9 位产品 → 看它的名称和各规范单位的**金额比重** → 点某个单位 → 看该单位下的**价格分布范围**。

**数据来源**（都在 `profile/`）

| 文件 | 用途 | 谁产的 |
|---|---|---|
| `product_names.csv` | 9 位码 → 产品名称 | 从 `code/description/bianma.dta` 导出 |
| `product_unit_matrix_{buy,sell}.csv` | 产品 × 规范单位 的金额/记录数 | `04_apply_unit_map.do` |
| `price_dist_{buy,sell}.csv` | 每个 (产品,单位) 的价格分位数 | `05_export_price_dist.do` |

价格分布必须在**企业层**观测上算，矩阵里只有聚合值，所以单独由 05 导出。
如果 `price_dist_*.csv` 还没同步回来，看板照样能用，只是价格面板会提示先跑 05。

**依赖**：`pandas` `matplotlib` `ipywidgets`

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline

# 中文字体：不设的话图里的中文全是方框
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
pd.set_option('display.max_rows', 60)

# profile 目录：notebook 放在 unit_harmonization/ 下，数据在它的 profile/ 子目录
HERE = Path.cwd()
for cand in (HERE / 'profile', HERE.parent / 'profile', HERE):
    if (cand / 'product_unit_matrix_buy.csv').exists():
        PROF = cand
        break
else:
    raise FileNotFoundError('找不到 product_unit_matrix_buy.csv，请把 notebook 放在 unit_harmonization/ 下运行')
print('PROF =', PROF)

# ---- 产品名称 ----
_n = pd.read_csv(PROF / 'product_names.csv', dtype={'product_id': str}, encoding='utf-8-sig')
NAME = dict(zip(_n['product_id'].str.strip(), _n['product_name'].str.strip()))
print(f'产品名称: {len(NAME)} 条')

# ---- 产品 × 单位 矩阵 + 价格分布 ----
MATRIX, PRICE = {}, {}
for side in ('buy', 'sell'):
    MATRIX[side] = pd.read_csv(PROF / f'product_unit_matrix_{side}.csv', dtype={'product_id': str})
    f = PROF / f'price_dist_{side}.csv'
    PRICE[side] = pd.read_csv(f, dtype={'product_id': str}) if f.exists() else None
    n = len(MATRIX[side])
    p = '已加载' if PRICE[side] is not None else '缺失（先跑 05_export_price_dist.do）'
    print(f'{side}: 矩阵 {n:,} 行 ｜ 价格分布 {p}')

In [ ]:
PCTS = [('p_min', 'min'), ('p1', '1%'), ('p5', '5%'), ('p10', '10%'), ('p25', '25%'),
        ('p50', '50%'), ('p75', '75%'), ('p90', '90%'), ('p95', '95%'), ('p99', '99%'),
        ('p_max', 'max')]

def fmt_money(v):
    """金额转成 万亿/亿/万 的可读形式"""
    v = float(v)
    for unit, div in (('万亿', 1e12), ('亿', 1e8), ('万', 1e4)):
        if abs(v) >= div:
            return f'{v / div:,.2f} {unit}'
    return f'{v:,.0f}'

def fmt_price(v):
    v = float(v)
    if not np.isfinite(v):
        return '—'
    if v >= 1000:
        return f'{v:,.0f}'
    if v >= 1:
        return f'{v:,.2f}'
    return f'{v:.4g}'

def product_options(side, kw=''):
    """下拉框选项，按金额降序；kw 同时匹配产品码和名称"""
    tot = MATRIX[side].groupby('product_id')['value'].sum().sort_values(ascending=False)
    kw = (kw or '').strip()
    opts = []
    for pid, v in tot.items():
        nm = NAME.get(pid, '(无名称)')
        if kw and (kw not in pid) and (kw not in nm):
            continue
        opts.append((f'{pid}  {nm}   [{fmt_money(v)}]', pid))
    return opts

def unit_table(side, pid):
    """某产品的各单位金额，降序"""
    d = MATRIX[side][MATRIX[side]['product_id'] == pid].copy()
    d = d.sort_values('value', ascending=False)
    d['share_pct'] = d['share'] * 100
    return d

def price_row(side, pid, unit):
    P = PRICE[side]
    if P is None:
        return None
    q = P[(P['product_id'] == pid) & (P['unit_std'] == unit)]
    return None if q.empty else q.iloc[0]

In [ ]:
import ipywidgets as W
from IPython.display import display, clear_output, Markdown

w_side = W.ToggleButtons(options=[('买方 buy', 'buy'), ('卖方 sell', 'sell')],
                         value='buy', description='口径')
w_kw   = W.Text(value='', description='搜索', continuous_update=False,
                placeholder='产品码或名称关键词，回车生效，如  钢  或  1080301',
                layout=W.Layout(width='680px'))
w_prod = W.Dropdown(options=product_options('buy'), description='产品',
                    layout=W.Layout(width='680px'))
w_unit = W.ToggleButtons(options=[], description='单位',
                         layout=W.Layout(width='100%'))
out_p, out_u = W.Output(), W.Output()

_busy = {'on': False}


def render_product():
    pid, side = w_prod.value, w_side.value
    with out_p:
        clear_output(wait=True)
        if pid is None:
            print('没有匹配的产品，换个关键词')
            return
        d = unit_table(side, pid)
        tot = d['value'].sum()
        top = d.iloc[0]
        display(Markdown(
            f"### {pid} &nbsp;&nbsp; {NAME.get(pid, '(无名称)')}\n"
            f"**{'买方' if side == 'buy' else '卖方'}口径** &nbsp;｜&nbsp; "
            f"总金额 **{fmt_money(tot)}** &nbsp;｜&nbsp; "
            f"用了 **{len(d)}** 个规范单位 &nbsp;｜&nbsp; "
            f"主导单位 **{top['unit_std']}**（{top['share_pct']:.1f}%）"))

        t = d[['unit_std', 'dimension', 'nrec', 'value', 'share_pct']].copy()
        t['value'] = t['value'].map(fmt_money)
        t['share_pct'] = t['share_pct'].map(lambda x: f'{x:.3f}')
        t.columns = ['单位', '量纲', '记录数', '金额', '金额比重%']
        display(t.reset_index(drop=True))

        k = d.head(12).iloc[::-1]
        fig, ax = plt.subplots(figsize=(8, max(2.4, 0.4 * len(k))))
        ax.barh(k['unit_std'], k['share_pct'], color='#4C78A8')
        for y, s in enumerate(k['share_pct']):
            ax.text(s, y, f'  {s:.2f}%', va='center', fontsize=9)
        ax.set_xlabel('占该产品金额的比重 (%)')
        ax.set_title('各单位的金额比重（前 12）')
        ax.margins(x=0.18)
        fig.tight_layout()
        plt.show()


def render_unit(*_):
    pid, side, u = w_prod.value, w_side.value, w_unit.value
    with out_u:
        clear_output(wait=True)
        if pid is None or u is None:
            return
        d = unit_table(side, pid)
        row = d[d['unit_std'] == u]
        if row.empty:
            return
        row = row.iloc[0]
        p_agg = row['value'] / row['qty_std'] if row['qty_std'] else np.nan

        display(Markdown(f"#### {pid} / 单位 **{u}** &nbsp;（{row['dimension']}）"))

        r = price_row(side, pid, u)
        if r is None:
            display(Markdown(
                f"聚合价 Σ金额/Σ数量 = **{fmt_price(p_agg)}** 元/{u}\n\n"
                f"> 跨企业的价格分布还没有数据。在 VM 上跑 `05_export_price_dist.do`，"
                f"把 `profile/price_dist_{side}.csv` 同步回来，这里就会显示完整分布。"))
            return

        display(Markdown(
            f"占本产品金额 **{r['share'] * 100:.2f}%** &nbsp;｜&nbsp; "
            f"观测 **{int(r['n_obs']):,}** 条 / **{int(r['n_firms']):,}** 家企业\n\n"
            f"聚合价（金额加权）**{fmt_price(r['p_agg'])}** &nbsp;｜&nbsp; "
            f"中位价（不加权）**{fmt_price(r['p50'])}** &nbsp;｜&nbsp; "
            f"CV **{r['cv']:.2f}** &nbsp;｜&nbsp; "
            f"p90/p10 **{r['r_p90p10']:,.1f}×** &nbsp;｜&nbsp; "
            f"p75/p25 **{r['r_p75p25']:,.1f}×**"))

        tab = pd.DataFrame({'分位': [lab for _, lab in PCTS],
                            f'价格（元/{u}）': [fmt_price(r[c]) for c, _ in PCTS]})
        display(tab)

        # 该单位的价格分布：箱线（p25-p75 盒 / p10-p90 须 / min-max 端点）
        fig, ax = plt.subplots(figsize=(9, 2.3))
        ax.plot([r['p10'], r['p90']], [0, 0], color='#999', lw=1.6, zorder=1)
        ax.add_patch(plt.Rectangle((r['p25'], -0.17), max(r['p75'] - r['p25'], 1e-12), 0.34,
                                   color='#4C78A8', alpha=0.65, zorder=2))
        ax.plot([r['p50']] * 2, [-0.22, 0.22], color='k', lw=2.2, zorder=3, label='中位价 p50')
        ax.scatter([r['p_agg']], [0], marker='D', s=60, color='#E45756', zorder=4,
                   label='聚合价（金额加权）')
        ax.scatter([r['p_min'], r['p_max']], [0, 0], marker='|', s=160, color='#bbb',
                   zorder=2, label='min / max')
        ax.set_xscale('log')
        ax.set_yticks([])
        ax.set_ylim(-0.6, 0.6)
        ax.set_xlabel(f'价格（元/{u}，对数轴）')
        ax.set_title(f'盒 = p25–p75　须 = p10–p90')
        ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.55), ncol=3, frameon=False, fontsize=8)
        fig.tight_layout()
        plt.show()

        # 同产品各单位横向对比：只看占比 >=1% 的
        P = PRICE[side]
        cmp = P[(P['product_id'] == pid) & (P['share'] >= 0.01)].sort_values('value', ascending=False)
        if len(cmp) > 1:
            cmp = cmp.iloc[::-1]
            fig, ax = plt.subplots(figsize=(9, max(2.4, 0.42 * len(cmp))))
            for y, (_, c) in enumerate(cmp.iterrows()):
                hit = (c['unit_std'] == u)
                ax.plot([c['p10'], c['p90']], [y, y], lw=6,
                        color='#E45756' if hit else '#c9d3e0',
                        solid_capstyle='butt', zorder=2)
                ax.scatter([c['p50']], [y], marker='|', s=180,
                           color='k' if hit else '#666', zorder=3)
            ax.set_yticks(range(len(cmp)))
            ax.set_yticklabels([f"{c['unit_std']}  {c['share'] * 100:.1f}%"
                                for _, c in cmp.iterrows()])
            ax.set_xscale('log')
            ax.set_xlabel('价格（对数轴）　条 = p10–p90，竖线 = 中位价')
            ax.set_title('同产品各单位的价格区间对比（仅占比 ≥1%，红色为当前选中）')
            fig.tight_layout()
            plt.show()


def refresh_units(*_):
    if _busy['on']:
        return
    pid = w_prod.value
    if pid is None:
        w_unit.options = []
        render_product()
        return
    d = unit_table(w_side.value, pid)
    _busy['on'] = True
    w_unit.options = [(f"{r.unit_std}  {r.share_pct:.1f}%", r.unit_std) for r in d.itertuples()]
    _busy['on'] = False
    render_product()
    if len(w_unit.options):
        w_unit.value = w_unit.options[0][1]
        render_unit()


def refresh_products(*_):
    opts = product_options(w_side.value, w_kw.value)
    _busy['on'] = True
    w_prod.options = opts
    _busy['on'] = False
    w_prod.value = opts[0][1] if opts else None
    refresh_units()


w_side.observe(lambda c: refresh_products(), names='value')
w_kw.observe(lambda c: refresh_products(), names='value')
w_prod.observe(refresh_units, names='value')
w_unit.observe(render_unit, names='value')

refresh_units()

display(W.VBox([w_side, w_kw, w_prod, out_p,
                W.HTML('<hr style="margin:14px 0">'),
                W.HTML('<b>点一个单位看它的价格分布 ↓</b>'),
                w_unit, out_u]))

## 不用交互控件的备用查法

`ipywidgets` 没装或者在纯 Jupyter 之外打开时，直接调函数。

In [ ]:
def show(pid, unit=None, side='buy'):
    """不依赖控件：show('108030100') 或 show('108030100', '千克')"""
    d = unit_table(side, pid)
    if d.empty:
        print(f'{side} 侧没有产品 {pid}')
        return
    print(f"{pid}  {NAME.get(pid, '(无名称)')}   {side} 侧   "
          f"总金额 {fmt_money(d['value'].sum())}   {len(d)} 个单位")
    t = d[['unit_std', 'dimension', 'nrec', 'value', 'share_pct']].copy()
    t['value'] = t['value'].map(fmt_money)
    t['share_pct'] = t['share_pct'].map(lambda x: f'{x:.3f}')
    t.columns = ['单位', '量纲', '记录数', '金额', '金额比重%']
    display(t.reset_index(drop=True))

    if unit is None:
        return
    r = price_row(side, pid, unit)
    if r is None:
        print(f'没有 {unit} 的价格分布（price_dist_{side}.csv 缺失或该单位不存在）')
        return
    print(f"\n{pid} / {unit}：{int(r['n_obs']):,} 条观测 / {int(r['n_firms']):,} 家企业   "
          f"聚合价 {fmt_price(r['p_agg'])}   中位价 {fmt_price(r['p50'])}   "
          f"p90/p10 {r['r_p90p10']:,.1f}×")
    display(pd.DataFrame({'分位': [l for _, l in PCTS],
                          f'价格（元/{unit}）': [fmt_price(r[c]) for c, _ in PCTS]}))


# 例：买方侧最大的钢材类产品
show('108030100', '千克')